# 3. Portfolio Construction

Screening gave us a shortlist of high-quality ETFs. This chapter answers the
question every investor actually cares about: **how much money goes where?**

The construction process has four stages:

1. **Strategic allocation** — start with fixed target weights per asset class
   (65% equities / 20% bonds / 5% precious metals / 10% commodities = 100%).
2. **[Sharpe](../content/99_glossary.md#risk-metrics) tilt** — nudge each ETF's weight up or
   down based on its recent risk-adjusted performance relative to the median
   ETF in its asset class.
3. **Volatility adjustment** — trim allocations to higher-volatility names so
   the portfolio's total risk stays consistent. Benchmark data is used here
   to calibrate cross-class volatility and cash weights.
4. **Cash allocation** — translate final weights into pound amounts against
   the annual £20,000 ISA allowance.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import load_intermediate, save_output
from etf_utils.metrics import calculate_annualized_volatility, interpolate_adjustment_factor
from etf_utils.database import load_screened_etfs, save_portfolio, seed_2025_portfolio

provider = DataProvider()

# Seed the locked 2025 portfolio (no-op if already done)
#seed_2025_portfolio()

# Load screened ETFs for 2026 from the DB (falls back to summary_all.csv if DB is empty)
etfs_df = load_screened_etfs(portfolio_year=2026)
if etfs_df.empty:
    etfs_df = pd.read_csv(DATA_INTERMEDIATE / 'summary_all.csv')

# Guard: drop any stale asset classes left over from a prior portfolio model run
# (e.g. 'energy' / 'agri' rows from the old 5-class model).  Only the four current
# classes are valid; anything else would cause a KeyError in sr_data_map below.
_EXPECTED_CLASSES = {'equity', 'bonds', 'preciousMetals', 'commodities'}
_stale = set(etfs_df['asset_class'].unique()) - _EXPECTED_CLASSES
if _stale:
    import warnings
    warnings.warn(
        f"Dropping stale asset classes from screened ETFs: {sorted(_stale)}. "
        "Re-run notebook 02 to refresh the DB, or manually delete via "
        "etf_utils.database.purge_screened_etfs_for_year(2026).",
        stacklevel=1,
    )
    etfs_df = etfs_df[etfs_df['asset_class'].isin(_EXPECTED_CLASSES)].copy()

print(f"Loaded {len(etfs_df)} ETFs for portfolio construction (asset classes: {etfs_df['asset_class'].unique().tolist()})")
etfs_df

Loaded 20 ETFs for portfolio construction (asset classes: ['equity', 'bonds', 'preciousMetals', 'commodities'])


,wkn,ticker,valor,name,inception_date,age_in_days,age_in_years,strategy,domicile_country,currency,...,available_on_investengine,yday_close_price_date,yday_close_price,metal_type,commodity_overlap_pct,within_metal_beta,tracking_fidelity,sector,benchmark_return,platform
0,A0HGV6,IUKD,2308843.0,iShares UK Dividend UCITS ETF,2005-11-04 00:00:00,7466.0,20.454795,Long-only,Ireland,GBP,...,None,2026-04-17,9.9940,NaN,None,None,None,None,None,InvestEngine
1,A1W9KD,XSTC,38111441.0,Xtrackers MSCI USA Information Technology UCIT...,2017-09-12 00:00:00,3136.0,8.591781,Long-only,Ireland,USD,...,None,2026-04-17,108.2400,NaN,None,None,None,None,None,InvestEngine
2,A0Q1YX,ISJP,4218299.0,iShares MSCI Japan Small Cap UCITS ETF (Dist),2008-05-09 00:00:00,6549.0,17.942466,Long-only,Ireland,USD,...,None,2026-04-17,43.1900,NaN,None,None,None,None,None,InvestEngine
3,A0MZWP,IMIB,3246482.0,iShares FTSE MIB UCITS ETF EUR (Dist),2007-07-06 00:00:00,6857.0,18.786301,Long-only,Ireland,EUR,...,None,2026-04-17,25.7950,NaN,None,None,None,None,None,InvestEngine
4,A2JF6S,VGER,41860972.0,Vanguard Germany All Cap UCITS ETF (EUR) Distr...,2018-07-17 00:00:00,2828.0,7.747945,Long-only,Ireland,EUR,...,None,2026-04-17,30.7550,NaN,None,None,None,None,None,InvestEngine
5,A0HGWA,IBZL,2308866.0,iShares MSCI Brazil UCITS ETF (Dist),2005-11-18 00:00:00,7452.0,20.416438,Long-only,Ireland,USD,...,None,2026-04-17,24.6725,NaN,None,None,None,None,None,InvestEngine
6,A1JHYT,HMCH,12534159.0,HSBC MSCI China UCITS ETF USD,2011-01-26 00:00:00,5557.0,15.224658,Long-only,Ireland,USD,...,None,2026-04-17,6.0425,NaN,None,None,None,None,None,InvestEngine
7,A1JJU5,HKOR,12843012.0,HSBC MSCI Korea Capped UCITS ETF USD,2011-04-06 00:00:00,5487.0,15.032877,Long-only,Ireland,USD,...,None,2026-04-17,95.0000,NaN,None,None,None,None,None,InvestEngine
8,A0LGP9,IGLT,2803931.0,iShares Core UK Gilts UCITS ETF,2006-12-01 00:00:00,7074.0,19.380822,Long-only,Ireland,GBP,...,None,2026-04-17,9.8800,NaN,None,None,None,None,None,InvestEngine
9,A0DKL3,SLXX,1828476.0,iShares Core GBP Corporate Bond UCITS ETF,2004-03-29 00:00:00,8051.0,22.057534,Long-only,Ireland,GBP,...,None,2026-04-16,120.5000,NaN,None,None,None,None,None,InvestEngine


## How the Sharpe tilt works

The core idea is simple: reward ETFs that have been **earning more return per
unit of risk** relative to peers in the same asset class, and reduce exposure
to those that have not. The mechanism:

| [Sharpe](../content/99_glossary.md#risk-metrics) vs class median | Adjustment applied |
|---|---|
| Well above median | Weight scaled up — up to **×1.48** |
| At median | Weight unchanged (×1.0) |
| Well below median | Weight scaled down — as low as **×0.60** |

The scale is calibrated so even the weakest fund retains a floor allocation
(0.60 × target weight), preserving diversification rather than collapsing to
one winner. The ceiling (1.48×) ensures no single ETF dominates the class.

Sensitivity — how much a given Sharpe deviation moves the multiplier — differs
by asset class because Sharpe distributions are tighter in equities (where
most funds cluster near the same broad index) than in bonds or commodities:

| Asset class | Sensitivity |
|---|---|
| Equities | ±0.10 |
| Bonds | ±0.25 |
| Precious metals & commodities | ±0.15 |

In [ ]:
# Sharpe ratio adjustment factors (top-level, across all asset classes)
trailing_sr_adjustment_factors = {
    -1.0: 0.60,
    -0.8: 0.66,
    -0.6: 0.77,
    -0.4: 0.85,
    -0.2: 0.94,
     0.0: 1.00,
     0.2: 1.11,
     0.4: 1.19,
     0.6: 1.30,
     0.8: 1.37,
     1.0: 1.48
}

# Initial risk weights — 4 asset classes: 65 / 20 / 5 / 10 (sums to 100)
eq_risk_weight  = 65   # Equities
bnd_risk_weight = 20   # Bonds
pm_risk_weight  = 5    # Precious Metals
cmd_risk_weight = 10   # Commodities (broad: energy ~30%, agri ~20%, metals)

# Regional risk weights for equity/bonds (unchanged)
regional_risk_weights_data = {
    'category': [
        'Developed_AmericasandUK',
        'Developed_EMEA',
        'Developed_APAC',
        'Emerging_Americas',
        'Emerging_APACandEMEA'
    ],
    'allocation': [10, 35, 35, 10, 10],
}
regional_risk_weights = pd.DataFrame(regional_risk_weights_data)

# Intra-asset Sharpe adjustment tables
# Equities: ±0.1 sensitivity
intra_asset_equity_trailing_sr_data = {
    -0.1: 0.6, -0.08: 0.66, -0.06: 0.77, -0.04: 0.85, -0.02: 0.94,
     0:   1.0,  0.02: 1.11,  0.04: 1.19,  0.06: 1.30,  0.08: 1.37, 0.1: 1.48
}

# Bonds: ±0.25 sensitivity
intra_asset_bond_trailing_sr_data = {
    -0.25: 0.6, -0.2: 0.66, -0.15: 0.77, -0.1: 0.85, -0.05: 0.94,
     0:    1.0,  0.05: 1.11,  0.1: 1.19,  0.15: 1.30,  0.2: 1.37, 0.25: 1.48
}

# Precious Metals: ±0.15 sensitivity
intra_asset_pm_trailing_sr_data = {
    -0.15: 0.6, -0.12: 0.66, -0.09: 0.77, -0.06: 0.85, -0.03: 0.94,
     0:    1.0,  0.03: 1.11,  0.06: 1.19,  0.09: 1.30,  0.12: 1.37, 0.15: 1.48
}

# Commodities: ±0.15 sensitivity
intra_asset_cmd_trailing_sr_data = {
    -0.15: 0.6, -0.12: 0.66, -0.09: 0.77, -0.06: 0.85, -0.03: 0.94,
     0:    1.0,  0.03: 1.11,  0.06: 1.19,  0.09: 1.30,  0.12: 1.37, 0.15: 1.48
}

# Map asset class → intra-asset SR table
sr_data_map = {
    'equity':         intra_asset_equity_trailing_sr_data,
    'bonds':          intra_asset_bond_trailing_sr_data,
    'preciousMetals': intra_asset_pm_trailing_sr_data,
    'commodities':    intra_asset_cmd_trailing_sr_data,
}


## Benchmark performance — cross-class reference

The benchmark **return and volatility** for each asset class are computed
first. They provide the cross-asset-class reference used later in the broader
calibration of the portfolio — for example, setting class-level cash weights
that account for how volatile the class has been.

The **intra-class tilt itself** is not benchmark-relative. Each ETF's Sharpe
is compared against the **median Sharpe of its own asset class** (the peer
median), and the adjustment multiplier is looked up from that delta.

The four benchmarks and their tickers are shown in the table above
([Screening → Benchmarks](02_etf_screening.ipynb)).

In [3]:
# Calculate benchmark returns and volatility for 2025 — all 4 asset classes
benchmark_year_start = "2025-01-01"
benchmark_year_end   = "2025-12-31"

def _benchmark_metrics(symbol, start, end):
    prices = provider.get_historical_prices(symbol, start_date=start, end_date=end)
    yr = prices["close"]
    ret = round((yr.iloc[-1] - yr.iloc[0]) / yr.iloc[0] * 100, 2)
    vol = round(calculate_annualized_volatility(yr) * 100, 2)
    sr  = round(ret / vol, 2)
    return ret, vol, sr

eq_benchmark_return,  eq_benchmark_volatility,  eq_sharpe_ratio  = _benchmark_metrics("VEVE", benchmark_year_start, benchmark_year_end)
bnd_benchmark_return, bnd_benchmark_volatility, bond_sharpe_ratio = _benchmark_metrics("SAAA", benchmark_year_start, benchmark_year_end)
pm_benchmark_return,  pm_benchmark_volatility,  pm_sharpe_ratio  = _benchmark_metrics("SGLN", benchmark_year_start, benchmark_year_end)
cmd_benchmark_return, cmd_benchmark_volatility, cmd_sharpe_ratio = _benchmark_metrics("CMOP", benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CMOP  return: {cmd_benchmark_return}%, vol: {cmd_benchmark_volatility}%, Sharpe: {cmd_sharpe_ratio}")

2025 VEVE  return: 12.76%,  vol: 14.71%,  Sharpe: 0.87
2025 SAAA  return: 2.82%, vol: 4.94%, Sharpe: 0.57
2025 SGLN  return: 48.91%,  vol: 17.65%,  Sharpe: 2.77
2025 CMOP  return: 5.56%, vol: 13.46%, Sharpe: 0.41


## Weight interpolation — the adjustment curve

The mapping from a Sharpe ratio to a weight multiplier is a **linear
interpolation** over a calibrated breakpoint table. The same curve applies
to all four asset classes; only the sensitivity parameter (±0.10 to ±0.25)
differs. Breakpoints at the extremes clamp the multiplier so it never falls
below 0.60 or rises above 1.48 regardless of outlier Sharpe values.

In [4]:
# interpolate_adjustment_factor is imported from etf_utils.metrics
# Example usage:
# factor = interpolate_adjustment_factor(sharpe_ratio, trailing_sr_adjustment_factors)

## Final portfolio

The steps below assemble the complete portfolio:

1. Compute each ETF's [Sharpe](../content/99_glossary.md#risk-metrics) relative to its
   asset-class median and look up the weight multiplier.
2. Multiply the strategic class weight by the multiplier and normalise within
   each class so the class still sums to its target allocation.
3. Apply the cross-class strategic split (65 / 20 / 5 / 10) and normalise to
   100%.
4. Scale by the annual ISA deposit (£20,000) to get the GBP investment per
   line.

The output table shows the key columns for review. A backup is written to
`data/output/final_portfolio.csv`; the authoritative copy lives in the
`portfolios` database table under year = 2026.

In [ ]:
# ── Top-level asset-class weight normalisation (4 classes) ──────────────────────
eq_adj_factor  = interpolate_adjustment_factor(eq_sharpe_ratio,  trailing_sr_adjustment_factors)
bond_adj_factor = interpolate_adjustment_factor(bond_sharpe_ratio, trailing_sr_adjustment_factors)
pm_adj_factor  = interpolate_adjustment_factor(pm_sharpe_ratio,  trailing_sr_adjustment_factors)
cmd_adj_factor = interpolate_adjustment_factor(cmd_sharpe_ratio, trailing_sr_adjustment_factors)

eq_adj  = eq_risk_weight  * eq_adj_factor
bnd_adj = bnd_risk_weight * bond_adj_factor
pm_adj  = pm_risk_weight  * pm_adj_factor
cmd_adj = cmd_risk_weight * cmd_adj_factor

total_adj = eq_adj + bnd_adj + pm_adj + cmd_adj
normalized_eq_weight   = round(eq_adj  / total_adj * 100, 2)
normalized_bond_weight = round(bnd_adj / total_adj * 100, 2)
normalized_pm_weight   = round(pm_adj  / total_adj * 100, 2)
normalized_cmd_weight  = round(cmd_adj / total_adj * 100, 2)
total_normalized_weight = normalized_eq_weight + normalized_bond_weight + normalized_pm_weight + normalized_cmd_weight

# Volatility-adjusted cash weights per asset class
eq_cash_weights  = normalized_eq_weight  / eq_benchmark_volatility
bond_cash_weights = normalized_bond_weight / bnd_benchmark_volatility
pm_cash_weights  = normalized_pm_weight  / pm_benchmark_volatility
cmd_cash_weights = normalized_cmd_weight / cmd_benchmark_volatility

total_cash_weight     = eq_cash_weights + bond_cash_weights + pm_cash_weights + cmd_cash_weights
final_eq_cash_weight   = total_normalized_weight * eq_cash_weights  / total_cash_weight
final_bond_cash_weight = total_normalized_weight * bond_cash_weights / total_cash_weight
final_pm_cash_weight   = total_normalized_weight * pm_cash_weights  / total_cash_weight
final_cmd_cash_weight  = total_normalized_weight * cmd_cash_weights / total_cash_weight

print("Asset Class Cash Weights (volatility-adjusted):")
print(f"  Equities:        {final_eq_cash_weight:.2f}%")
print(f"  Bonds:           {final_bond_cash_weight:.2f}%")
print(f"  Precious Metals: {final_pm_cash_weight:.2f}%")
print(f"  Commodities:     {final_cmd_cash_weight:.2f}%")
total_check = final_eq_cash_weight + final_bond_cash_weight + final_pm_cash_weight + final_cmd_cash_weight
print(f"  Total:           {total_check:.2f}%")

In [ ]:
etfs = etfs_df.copy()

# ── Region category weight ──────────────────────────────────────────────────────
# Precious Metals and Commodities use a single Global region → 100% weight
_regional_lookup = regional_risk_weights.set_index('category')['allocation']

def _get_region_weight(row):
    if row['region_category'] == 'Global':
        return 100
    return _regional_lookup.get(row['region_category'], 0)

etfs['region_category_weight'] = etfs.apply(_get_region_weight, axis=1)

# ── Intra-region category weight ───────────────────────────────────────────────
etfs['intra_region_category_weight'] = 0

for asset in etfs['asset_class'].unique():
    for region_category in etfs[etfs['asset_class'] == asset]['region_category'].unique():
        mask = (etfs['asset_class'] == asset) & (etfs['region_category'] == region_category)
        num_categories = len(etfs[mask]['intra_region_category'].unique())
        if num_categories > 0:
            etfs.loc[mask, 'intra_region_category_weight'] = 100 // num_categories

# ── Sharpe ratio calculations ──────────────────────────────────────────────────
etfs['starting_risk_weights'] = etfs['region_category_weight'] * etfs['intra_region_category_weight'] / 100
etfs['yield'] = (etfs['last_year_dividends'] - etfs['ter']).round(2)
etfs['sharpe_ratio'] = (etfs['yield'] / etfs['last_year_volatility']).round(2)

# Accumulating ETCs (PM, commodities) have no dividend → last_year_dividends is NaN
# → sharpe_ratio would also be NaN, silently zeroing all downstream risk weights.
# Fall back to last_year_return_per_risk which is already a risk-adjusted return.
etfs['sharpe_ratio'] = etfs['sharpe_ratio'].fillna(etfs['last_year_return_per_risk'])

# Median Sharpe per asset class
median_sharpe = {
    ac: etfs[etfs['asset_class'] == ac]['sharpe_ratio'].median()
    for ac in etfs['asset_class'].unique()
}

etfs['relative_sharpe_ratio'] = etfs.apply(
    lambda row: row['sharpe_ratio'] - median_sharpe[row['asset_class']], axis=1
)
etfs['interpolated_adjusted_factors'] = etfs.apply(
    lambda row: interpolate_adjustment_factor(
        row['relative_sharpe_ratio'], sr_data_map[row['asset_class']]
    ), axis=1
)

# ── Risk weights ───────────────────────────────────────────────────────────────
etfs['adjusted_risk_weights'] = (
    etfs['interpolated_adjusted_factors'] * etfs['starting_risk_weights']
).round(2)

# Normalize within each asset class to sum to 100%
etfs['normalized_risk_weights'] = etfs.groupby('asset_class')['adjusted_risk_weights'].transform(
    lambda x: x / x.sum() * 100
)

# ── Cash weights (intra-asset class) ──────────────────────────────────────────
etfs['cash_weights_guess'] = etfs['normalized_risk_weights'] / etfs['last_year_volatility']

etfs['cash_weights'] = etfs.groupby('asset_class')['cash_weights_guess'].transform(
    lambda x: (etfs.loc[x.index, 'normalized_risk_weights'].sum() / x.sum()) * x
).round(2)

# Final regional category weight (sum of cash weights per region per asset class)
etfs['region_category_weight_final'] = etfs.groupby(
    ['asset_class', 'region_category']
)['cash_weights'].transform('sum')

# ── Asset class weight map (4 classes) ────────────────────────────────────────
asset_class_weight_map = {
    'equity':         normalized_eq_weight,
    'bonds':          normalized_bond_weight,
    'preciousMetals': normalized_pm_weight,
    'commodities':    normalized_cmd_weight,
}
etfs['normalized_asset_class_weight'] = etfs['asset_class'].map(asset_class_weight_map)

# ── Final weights (all 4 asset classes together) ──────────────────────────────
etfs['final_risk_weights'] = etfs['normalized_asset_class_weight'] * etfs['normalized_risk_weights'] / 100
etfs['final_cash_weights_guess'] = etfs['final_risk_weights'] / etfs['last_year_volatility']
etfs['final_cash_weights'] = (
    etfs['final_risk_weights'].sum() / etfs['final_cash_weights_guess'].sum()
    * etfs['final_cash_weights_guess']
).round()

# ── Investment amounts (GBP) ───────────────────────────────────────────────────
etfs['investment'] = 20000 * etfs['final_cash_weights'] / 100

# ── Save to DB (year=2026, overwritable) and CSV backup ──────────────────────
save_portfolio(etfs, year=2026)
etfs.to_csv(DATA_OUTPUT / 'final_portfolio.csv', index=False)

# 2025 calendar-year return sparse in equity/bond raw data; fall back to
# trailing 12-month return (last_year) so the display column is always populated.
etfs['2025'] = etfs['2025'].fillna(etfs['last_year'])

(
    etfs
    .sort_values(by=['asset_class', 'investment'], ascending=[False, False])
    [[
        'asset_class', 'ticker', 'name', 'ter',
        'final_cash_weights', 'investment', 'platform',
    ]]
    .rename(columns={
        'asset_class': 'Class',
        'ticker': 'Ticker',
        'name': 'Name',
        'ter': 'TER (%)',
        'final_cash_weights': 'Weight (%)',
        'investment': 'Investment (£)',
        'platform': 'Platform',
    })
    .style.format({
        'TER (%)': '{:.2f}',
        'Weight (%)': '{:.1f}',
        'Investment (£)': '£{:,.0f}',
    })
)